# Camada Silver — Students Performance

**Arquitetura Medalhão — Camada Silver**

Responsabilidade: limpeza, validação e enriquecimento dos dados brutos da camada Bronze.  
Os dados são padronizados, têm tipos corretos, duplicatas removidas e colunas categóricas decodificadas.

**Decisões de qualidade:**
- A coluna `Ethnicity` é **removida** nesta camada por dois motivos:
  1. **LGPD**: trata-se de dado sensível que exige tratamento especial.
  2. **Fairness/Equidade Algorítmica**: a variável apresenta forte desbalanceamento entre categorias, o que poderia gerar viés no modelo preditivo.
- Colunas categóricas são decodificadas para legibilidade humana (mantendo os valores originais).

**Fonte:** tabela `bronze_students`  
**Destino:** tabela `silver_students` no SQLite (`data_lakehouse.db`)

In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [2]:
# Conexão com o banco de dados SQLite (lakehouse local)
conn = sqlite3.connect("../../data/data_lakehouse.db")

## 1. Leitura da camada Bronze

In [3]:
df = pd.read_sql("SELECT * FROM bronze_students", conn)
print(f"Shape Bronze: {df.shape}")
df.head()

Shape Bronze: (2392, 15)


,StudentID,Age,Gender,Ethnicity,ParentalEducation,StudyTimeWeekly,Absences,Tutoring,ParentalSupport,Extracurricular,Sports,Music,Volunteering,GPA,GradeClass
0,1001,17,1,0,2,19.833723,7,1,2,0,0,1,0,2.929196,2.0
1,1002,18,0,0,1,15.408756,0,0,1,0,0,0,0,3.042915,1.0
2,1003,15,0,2,3,4.210570,26,0,2,0,0,0,0,0.112602,4.0
3,1004,17,1,0,3,10.028829,14,0,3,1,0,0,0,2.054218,3.0
4,1005,17,1,0,2,4.672495,17,1,3,0,0,0,0,1.288061,4.0


## 2. Remoção da coluna Ethnicity (LGPD + Fairness)

In [4]:
# Remoção justificada da coluna Ethnicity:
# - Dado sensível conforme LGPD (Lei 13.709/2018)
# - Desbalanceamento acentuado entre categorias (risco de viés algorítmico)
print("Distribuição de Ethnicity na Bronze (antes da remoção):")
print(df["Ethnicity"].value_counts().to_string())
print()

df = df.drop(columns=["Ethnicity"])
print(f"Coluna 'Ethnicity' removida. Shape após remoção: {df.shape}")

Distribuição de Ethnicity na Bronze (antes da remoção):
Ethnicity
0    1207
1     493
2     470
3     222

Coluna 'Ethnicity' removida. Shape após remoção: (2392, 14)


## 3. Validação e correção de tipos

In [5]:
# Tipos esperados
int_cols = ["StudentID", "Age", "Gender", "ParentalEducation",
            "Tutoring", "ParentalSupport", "Extracurricular",
            "Sports", "Music", "Volunteering"]
float_cols = ["StudyTimeWeekly", "GPA"]

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# GradeClass pode vir como float (ex: 2.0) — converter para int
df["GradeClass"] = pd.to_numeric(df["GradeClass"], errors="coerce").astype("Int64")

print("Tipos após conversão:")
print(df.dtypes.to_string())

Tipos após conversão:
StudentID              Int64
Age                    Int64
Gender                 Int64
ParentalEducation      Int64
StudyTimeWeekly      float64
Absences               int64
Tutoring               Int64
ParentalSupport        Int64
Extracurricular        Int64
Sports                 Int64
Music                  Int64
Volunteering           Int64
GPA                  float64
GradeClass             Int64


## 4. Tratamento de valores nulos e duplicatas

In [6]:
# Verificação de nulos após conversão de tipos
nulos = df.isnull().sum()
print("Valores nulos por coluna:")
print(nulos.to_string())

linhas_antes = len(df)
# Remoção de linhas com StudentID nulo ou GPA nulo (campos essenciais)
df = df.dropna(subset=["StudentID", "GPA", "GradeClass"])
print(f"\nLinhas removidas por nulos em campos essenciais: {linhas_antes - len(df)}")

Valores nulos por coluna:
StudentID            0
Age                  0
Gender               0
ParentalEducation    0
StudyTimeWeekly      0
Absences             0
Tutoring             0
ParentalSupport      0
Extracurricular      0
Sports               0
Music                0
Volunteering         0
GPA                  0
GradeClass           0

Linhas removidas por nulos em campos essenciais: 0


In [7]:
# Remoção de duplicatas por StudentID (chave primária)
n_antes = len(df)
df = df.drop_duplicates(subset=["StudentID"])
print(f"Duplicatas removidas por StudentID: {n_antes - len(df)}")
print(f"Shape após limpeza: {df.shape}")

Duplicatas removidas por StudentID: 0
Shape após limpeza: (2392, 14)


## 5. Validação de intervalos esperados

In [8]:
# Validação de intervalos conforme documentação do dataset
checks = {
    "Age (15-18)": df["Age"].between(15, 18).all(),
    "Gender (0-1)": df["Gender"].isin([0, 1]).all(),
    "ParentalEducation (0-4)": df["ParentalEducation"].isin([0, 1, 2, 3, 4]).all(),
    "StudyTimeWeekly (0-20)": df["StudyTimeWeekly"].between(0, 20).all(),
    "Absences (0-30)": df["Absences"].between(0, 30).all(),
    "GPA (0-4)": df["GPA"].between(0.0, 4.0).all(),
    "GradeClass (0-4)": df["GradeClass"].isin([0, 1, 2, 3, 4]).all(),
}

for check, resultado in checks.items():
    status = "✅ OK" if resultado else "⚠️ FALHOU"
    print(f"{status} — {check}")

✅ OK — Age (15-18)
✅ OK — Gender (0-1)
✅ OK — ParentalEducation (0-4)
✅ OK — StudyTimeWeekly (0-20)
✅ OK — Absences (0-30)
✅ OK — GPA (0-4)
✅ OK — GradeClass (0-4)


## 6. Decodificação de colunas categóricas

In [9]:
# Decodificação: mantém colunas numéricas originais e adiciona versões legíveis com sufixo _label

# Gender
df["Gender_label"] = df["Gender"].map({0: "Masculino", 1: "Feminino"})

# ParentalEducation
educacao_map = {
    0: "Nenhum",
    1: "Ensino Médio",
    2: "Faculdade Incompleta",
    3: "Bacharelado",
    4: "Pós-Graduação"
}
df["ParentalEducation_label"] = df["ParentalEducation"].map(educacao_map)

# ParentalSupport
suporte_map = {
    0: "Nenhum",
    1: "Baixo",
    2: "Moderado",
    3: "Alto",
    4: "Muito Alto"
}
df["ParentalSupport_label"] = df["ParentalSupport"].map(suporte_map)

# Tutoring, Extracurricular, Sports, Music, Volunteering
for col in ["Tutoring", "Extracurricular", "Sports", "Music", "Volunteering"]:
    df[f"{col}_label"] = df[col].map({0: "Não", 1: "Sim"})

# GradeClass
gradeclass_map = {
    0: "A (GPA >= 3.5)",
    1: "B (3.0 <= GPA < 3.5)",
    2: "C (2.5 <= GPA < 3.0)",
    3: "D (2.0 <= GPA < 2.5)",
    4: "F (GPA < 2.0)"
}
df["GradeClass_label"] = df["GradeClass"].map(gradeclass_map)

print(f"Colunas após enriquecimento: {list(df.columns)}")
df.head()

Colunas após enriquecimento: ['StudentID', 'Age', 'Gender', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA', 'GradeClass', 'Gender_label', 'ParentalEducation_label', 'ParentalSupport_label', 'Tutoring_label', 'Extracurricular_label', 'Sports_label', 'Music_label', 'Volunteering_label', 'GradeClass_label']


,StudentID,Age,Gender,ParentalEducation,StudyTimeWeekly,Absences,Tutoring,ParentalSupport,Extracurricular,Sports,...,GradeClass,Gender_label,ParentalEducation_label,ParentalSupport_label,Tutoring_label,Extracurricular_label,Sports_label,Music_label,Volunteering_label,GradeClass_label
0,1001,17,1,2,19.833723,7,1,2,0,0,...,2,Feminino,Faculdade Incompleta,Moderado,Sim,Não,Não,Sim,Não,C (2.5 <= GPA < 3.0)
1,1002,18,0,1,15.408756,0,0,1,0,0,...,1,Masculino,Ensino Médio,Baixo,Não,Não,Não,Não,Não,B (3.0 <= GPA < 3.5)
2,1003,15,0,3,4.210570,26,0,2,0,0,...,4,Masculino,Bacharelado,Moderado,Não,Não,Não,Não,Não,F (GPA < 2.0)
3,1004,17,1,3,10.028829,14,0,3,1,0,...,3,Feminino,Bacharelado,Alto,Não,Sim,Não,Não,Não,D (2.0 <= GPA < 2.5)
4,1005,17,1,2,4.672495,17,1,3,0,0,...,4,Feminino,Faculdade Incompleta,Alto,Sim,Não,Não,Não,Não,F (GPA < 2.0)


## 7. Persistência na camada Silver

In [10]:
# GradeClass como Int64 não é aceito pelo SQLite — converter para int padrão
int64_cols = df.select_dtypes(include=["Int64"]).columns.tolist()
for col in int64_cols:
    df[col] = df[col].astype(float).astype(int)

rows = df.to_sql("silver_students", conn, if_exists="replace", index=False)
print(f"Registros gravados em silver_students: {rows}")

Registros gravados em silver_students: 2392


In [11]:
# Validação final
count = pd.read_sql("SELECT COUNT(*) as total FROM silver_students", conn)
print(f"Total de registros em silver_students: {count['total'][0]}")

sample = pd.read_sql("SELECT * FROM silver_students LIMIT 5", conn)
sample

Total de registros em silver_students: 2392


,StudentID,Age,Gender,ParentalEducation,StudyTimeWeekly,Absences,Tutoring,ParentalSupport,Extracurricular,Sports,...,GradeClass,Gender_label,ParentalEducation_label,ParentalSupport_label,Tutoring_label,Extracurricular_label,Sports_label,Music_label,Volunteering_label,GradeClass_label
0,1001,17,1,2,19.833723,7,1,2,0,0,...,2,Feminino,Faculdade Incompleta,Moderado,Sim,Não,Não,Sim,Não,C (2.5 <= GPA < 3.0)
1,1002,18,0,1,15.408756,0,0,1,0,0,...,1,Masculino,Ensino Médio,Baixo,Não,Não,Não,Não,Não,B (3.0 <= GPA < 3.5)
2,1003,15,0,3,4.210570,26,0,2,0,0,...,4,Masculino,Bacharelado,Moderado,Não,Não,Não,Não,Não,F (GPA < 2.0)
3,1004,17,1,3,10.028829,14,0,3,1,0,...,3,Feminino,Bacharelado,Alto,Não,Sim,Não,Não,Não,D (2.0 <= GPA < 2.5)
4,1005,17,1,2,4.672495,17,1,3,0,0,...,4,Feminino,Faculdade Incompleta,Alto,Sim,Não,Não,Não,Não,F (GPA < 2.0)


In [12]:
conn.close()
print("Camada Silver concluída com sucesso.")

Camada Silver concluída com sucesso.
